In [15]:
"""
Fetch remote job postings from RemoteOK and Remotive and land them
into separate line-based CSV files for analysis.

Usage:
    pip install requests
    python fetch_jobs_to_excel.py

Output:
    remoteok_jobs_raw.csv
        - append one row per RemoteOK job on each run
    remotive_jobs_raw.csv
        - append one row per Remotive job on each run

Notes / API constraints respected:
    - RemoteOK: https://remoteok.com/api requires no key. First array
      element is a legal/metadata object, not a job - it is skipped.
      RemoteOK's terms require crediting/linking back to job URLs.
    - Remotive: https://remotive.com/api/remote-jobs requires no key
      for this basic usage. Remotive asks for at most ~4 requests/day,
      so this script is meant to be run on a daily schedule, not looped.
"""

import csv
from datetime import datetime, timezone

import requests

REMOTEOK_URL = "https://remoteok.com/api"
REMOTIVE_URL = "https://remotive.com/api/remote-jobs"
# A descriptive User-Agent is good practice for public APIs like these
HEADERS = {"User-Agent": "job-market-bi-project/1.0 (personal data pipeline)"}

REMOTEOK_FIELDS = [
    "id", "slug", "company", "position", "tags", "location",
    "salary_min", "salary_max", "date", "epoch", "apply_url", "url",
    "description",
]

REMOTIVE_FIELDS = [
    "id", "title", "company_name", "category", "job_type",
    "candidate_required_location", "salary", "publication_date", "url",
    "description",
]


def fetch_remoteok():
    resp = requests.get(REMOTEOK_URL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    # First element is a legal notice, not a job record - skip anything without an id
    jobs = [row for row in data if isinstance(row, dict) and row.get("id")]
    return jobs


def fetch_remotive():
    resp = requests.get(REMOTIVE_URL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data.get("jobs", [])


def normalize_value(val):
    # Flatten list fields (e.g. RemoteOK "tags") into a comma-separated string.
    if isinstance(val, list):
        val = ", ".join(str(v) for v in val)

    # Keep the export robust if a description is unusually long.
    if isinstance(val, str) and len(val) > 32767:
        val = val[:32750] + " ...[truncated: exceeded Excel's 32,767-char cell limit]"

    return val


def flatten_records(source_name, fields, records, fetched_at):
    rows = []
    for rec in records:
        row = {field: normalize_value(rec.get(field, "")) for field in fields}
        row["source"] = source_name
        row["fetched_at"] = fetched_at
        rows.append(row)
    return rows


def write_csv(out_name, fields, rows):
    header = ["source", "fetched_at"] + fields
    write_header = True

    try:
        with open(out_name, "r", newline="", encoding="utf-8") as existing:
            write_header = existing.read(1) == ""
    except FileNotFoundError:
        write_header = True

    with open(out_name, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=header, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


def main():
    fetched_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    errors = []

    try:
        remoteok_jobs = fetch_remoteok()
    except Exception as e:
        remoteok_jobs = []
        errors.append(f"RemoteOK fetch failed: {e}")

    try:
        remotive_jobs = fetch_remotive()
    except Exception as e:
        remotive_jobs = []
        errors.append(f"Remotive fetch failed: {e}")

    remoteok_out = "remoteok_jobs_raw.csv"
    remotive_out = "remotive_jobs_raw.csv"

    write_csv(
        remoteok_out,
        REMOTEOK_FIELDS,
        flatten_records("RemoteOK", REMOTEOK_FIELDS, remoteok_jobs, fetched_at),
    )
    write_csv(
        remotive_out,
        REMOTIVE_FIELDS,
        flatten_records("Remotive", REMOTIVE_FIELDS, remotive_jobs, fetched_at),
    )

    print(f"Saved {remoteok_out}")
    print(f"Saved {remotive_out}")
    if errors:
        print("Errors encountered:")
        for e in errors:
            print(" -", e)


if __name__ == "__main__":
    main()


Saved remoteok_jobs_raw.csv
Saved remotive_jobs_raw.csv
